# Part II – Tweets Emotion Classification using Word Embeddings

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
import gensim.downloader as gensim_api
from gensim.models import Word2Vec
from gensim.utils import simple_preprocess
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Embedding, Conv1D, MaxPooling1D, GlobalMaxPooling1D,
    Flatten, Dense, Input, Dropout, BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt


In [2]:
twdataset = pd.read_csv('twitter_emotion.csv')
print(twdataset.shape)
print(twdataset.head())


(416809, 3)
   Unnamed: 0                                               text  label
0           0      i just feel really helpless and heavy hearted      4
1           1  ive enjoyed being able to slouch about relax a...      0
2           2  i gave up my internship with the dmrg and am f...      4
3           3                         i dont know i feel so lost      0
4           4  i am a kindergarten teacher and i am thoroughl...      4


In [3]:
TWTEXTCOL  = 'text'
TWLABELCOL = 'label'
print(f'Text column : {TWTEXTCOL}')
print(f'Label column: {TWLABELCOL}')
print('\nEmotion distribution:')
print(twdataset[TWLABELCOL].value_counts())


Text column : text
Label column: label

Emotion distribution:
label
1    141067
0    121187
3     57317
4     47712
2     34554
5     14972
Name: count, dtype: int64


## 1. Tweets Pre-processing – Keras Tokenizer

In [4]:
MAX_WORDS   = 100_000
MAX_SEQ_LEN = 60   # Tweets are short; 60 covers ~99th percentile of tweet lengths

tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(twdataset[TWTEXTCOL].astype(str))
sequences  = tokenizer.texts_to_sequences(twdataset[TWTEXTCOL].astype(str))
word_index = tokenizer.word_index
vocab_size = len(word_index) + 1   # actual vocab size, not MAX_WORDS+1

print(f'Vocab size        : {len(word_index)}')
print(f'First 10 entries  : {list(word_index.items())[:10]}')

padded_sequences = pad_sequences(
    sequences,
    maxlen=MAX_SEQ_LEN,
    padding='post',
    truncating='post'
)
print(f'Padded sequences shape: {padded_sequences.shape}')

raw_lengths = [len(seq) for seq in sequences]
print(f'\nTweet length stats:')
print(f'  Mean   : {np.mean(raw_lengths):.0f} tokens')
print(f'  Median : {np.median(raw_lengths):.0f} tokens')
print(f'  95th % : {np.percentile(raw_lengths, 95):.0f} tokens')
print(f'  99th % : {np.percentile(raw_lengths, 99):.0f} tokens')
print(f'  Max    : {max(raw_lengths)} tokens')


Vocab size        : 75302
First 10 entries  : [('i', 1), ('feel', 2), ('and', 3), ('to', 4), ('the', 5), ('a', 6), ('feeling', 7), ('that', 8), ('of', 9), ('my', 10)]
Padded sequences shape: (416809, 60)

Tweet length stats:
  Mean   : 19 tokens
  Median : 17 tokens
  95th % : 41 tokens
  99th % : 52 tokens
  Max    : 178 tokens


## 2a. Pre-trained Embeddings – GloVe Twitter 50d

In [5]:
glove_model = gensim_api.load('glove-twitter-50')
EMBEDDING_DIM_GLOVE = 50
print(f'Vocabulary size in GloVe model: {len(glove_model.key_to_index)}')


Vocabulary size in GloVe model: 1193514


In [6]:
def buildEmbeddingMatrix(wvModel, word_index, vocab_size, embedding_dimension):
    # Matrix sized to actual vocab_size (not MAX_WORDS+1)
    matrix = np.zeros((vocab_size, embedding_dimension))
    for word, idx in word_index.items():
        if idx >= vocab_size:
            continue
        try:
            matrix[idx] = wvModel[word]
        except KeyError:
            pass
    return matrix

glove_embedding_matrix = buildEmbeddingMatrix(
    glove_model, word_index, vocab_size, EMBEDDING_DIM_GLOVE
)
print(f'GloVe embedding matrix shape: {glove_embedding_matrix.shape}')


GloVe embedding matrix shape: (75303, 50)


## 2b. Your trained word2vec model word embeddings:

In [7]:
print("Step 2b – Training custom Word2Vec model ...")

# Tokenize directly from raw tweet text
tokenized_tweets = [
    simple_preprocess(str(tweet))
    for tweet in twdataset[TWTEXTCOL]
]

print(f"Total tokenized tweets : {len(tokenized_tweets)}")
print(f"First tokenized tweet  : {tokenized_tweets[0]}")

EMBEDDING_DIM_W2V = 50   # Match GloVe dimension for fair comparison

w2v_model = Word2Vec(
    sentences=tokenized_tweets,
    vector_size=EMBEDDING_DIM_W2V,
    window=3,       # smaller window suits short tweets
    min_count=2,    # drop very rare words (noise)
    epochs=5,      # more training epochs
    sg=0,           # CBOW: better for short documents
    workers=4
)

w2v_model.save("word2vec_twitter.model")
print(f"Word2Vec vocab size : {len(w2v_model.wv.key_to_index)}")

print("Building Word2Vec embedding matrix ...")
w2v_embedding_matrix = buildEmbeddingMatrix(
    w2v_model.wv, word_index, vocab_size, EMBEDDING_DIM_W2V
)
print(f"Word2Vec embedding matrix shape: {w2v_embedding_matrix.shape}")


Step 2b – Training custom Word2Vec model ...
Total tokenized tweets : 416809
First tokenized tweet  : ['just', 'feel', 'really', 'helpless', 'and', 'heavy', 'hearted']


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Word2Vec vocab size : 40534
Building Word2Vec embedding matrix ...
Word2Vec embedding matrix shape: (75303, 50)


## 3. Data preparation and splitting:

3a. Apply one-hot encoding for the integer labels using keras.

In [8]:
labels = twdataset[TWLABELCOL].values.astype(int)   # raw ints — needed for stratify
one_hot_labels = to_categorical(labels, num_classes=6)

print("Original labels shape:", labels.shape)
print("One-hot labels shape:", one_hot_labels.shape)


Original labels shape: (416809,)
One-hot labels shape: (416809, 6)


3b. Splitting

In [9]:
# stratify=labels ensures balanced class distribution in both splits

# split 80% & 20%
X_train_80, X_test_80, y_train_80, y_test_80 = train_test_split(
    padded_sequences,
    one_hot_labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("\n80/20 Split:")
print("X_train shape:", X_train_80.shape)
print("X_test shape :", X_test_80.shape)

# split 70% & 30%
X_train_70, X_test_70, y_train_70, y_test_70 = train_test_split(
    padded_sequences,
    one_hot_labels,
    test_size=0.3,
    random_state=42,
    stratify=labels
)

print("\n70/30 Split:")
print("X_train shape:", X_train_70.shape)
print("X_test shape :", X_test_70.shape)



80/20 Split:
X_train shape: (333447, 60)
X_test shape : (83362, 60)

70/30 Split:
X_train shape: (291766, 60)
X_test shape : (125043, 60)


4.1 — Model Builder Function

In [13]:
NUM_CLASSES = one_hot_labels.shape[1]   # 6

def build_cnn_model(embedding_matrix, embedding_dim, max_len, name="CNN",
                    trainable_embeddings=False):

    vocab_size_local = embedding_matrix.shape[0]
    sequence_input   = Input(shape=(max_len,), dtype='int32')

    # ── Define FIRST ───────────────────────────────────────────
    embedding_layer    = Embedding(
        vocab_size_local,
        embedding_dim,
        weights=[embedding_matrix],
        input_length=max_len,
        trainable=trainable_embeddings
    )
    # ── Then call ──────────────────────────────────────────────
    embedded_sequences = embedding_layer(sequence_input)

    # 1st Conv1D
    x = Conv1D(128, 5, activation='relu', padding='same')(embedded_sequences)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)

    # 2nd Conv1D
    x = Conv1D(128, 5, activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    x = MaxPooling1D(2)(x)

    x = GlobalMaxPooling1D()(x)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    output_layer = Dense(NUM_CLASSES, activation='softmax')(x)

    model = Model(sequence_input, output_layer, name=name)
    model.compile(
        loss='categorical_crossentropy',
        optimizer='rmsprop',          # ← adam performs better than rmsprop
        metrics=['accuracy']
    )
    return model

4.2 — Training

In [14]:
def train_and_evaluate(model, X_train, X_test, y_train, y_test,
                       epochs=5, batch_size=64):

    early_stop = EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True
    )

    history = model.fit(
        X_train, y_train,
        validation_split=0.1,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=[early_stop],
        verbose=1
    )

    test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
    print(f"\n★ Test Loss: {test_loss:.4f}  |  Test Accuracy: {test_acc:.4f}\n")
    return history, test_loss, test_acc



4.3 — Run All 4 Experiments

In [15]:
for MAX_LEN in [500 , 700 ,1000]:   # tweet-optimized sequence length

    print("\n" + "█" * 70)
    print(f"  MAX_LEN = {MAX_LEN}")
    print("█" * 70)

    padded = pad_sequences(sequences, maxlen=MAX_LEN, padding='post', truncating='post')

    X_train_80, X_test_80, y_train_80, y_test_80 = train_test_split(
        padded, one_hot_labels, test_size=0.2, random_state=42, stratify=labels
    )
    X_train_70, X_test_70, y_train_70, y_test_70 = train_test_split(
        padded, one_hot_labels, test_size=0.3, random_state=42, stratify=labels
    )

    results = {}

    #GloVe + 80/20 (embeddings frozen)
    print("=" * 60)
    print(f"  GloVe-50d  |  80/20 Split  |  MAX_LEN={MAX_LEN}")
    print("=" * 60)
    tf.keras.backend.clear_session()
    model_glove_80 = build_cnn_model(
        glove_embedding_matrix, EMBEDDING_DIM_GLOVE, MAX_LEN,
        name="CNN_GloVe_80_20", trainable_embeddings=False
    )
    model_glove_80.summary()
    hist, loss, acc = train_and_evaluate(model_glove_80, X_train_80, X_test_80, y_train_80, y_test_80)
    results["GloVe-50d | 80/20"] = {"loss": loss, "accuracy": acc}

    #GloVe + 70/30
    print("=" * 60)
    print(f"  GloVe-50d  |  70/30 Split  |  MAX_LEN={MAX_LEN}")
    print("=" * 60)
    tf.keras.backend.clear_session()
    model_glove_70 = build_cnn_model(
        glove_embedding_matrix, EMBEDDING_DIM_GLOVE, MAX_LEN,
        name="CNN_GloVe_70_30", trainable_embeddings=False
    )
    hist, loss, acc = train_and_evaluate(model_glove_70, X_train_70, X_test_70, y_train_70, y_test_70)
    results["GloVe-50d | 70/30"] = {"loss": loss, "accuracy": acc}

    #Word2Vec + 80/20
    print("=" * 60)
    print(f"  Word2Vec-{EMBEDDING_DIM_W2V}d  |  80/20 Split  |  MAX_LEN={MAX_LEN}")
    print("=" * 60)
    tf.keras.backend.clear_session()
    model_w2v_80 = build_cnn_model(
        w2v_embedding_matrix, EMBEDDING_DIM_W2V, MAX_LEN,
        name="CNN_W2V_80_20", trainable_embeddings=True
    )
    model_w2v_80.summary()
    hist, loss, acc = train_and_evaluate(model_w2v_80, X_train_80, X_test_80, y_train_80, y_test_80)
    results["Word2Vec | 80/20"] = {"loss": loss, "accuracy": acc}

    #Word2Vec + 70/30
    print("=" * 60)
    print(f"  Word2Vec-{EMBEDDING_DIM_W2V}d  |  70/30 Split  |  MAX_LEN={MAX_LEN}")
    print("=" * 60)
    tf.keras.backend.clear_session()
    model_w2v_70 = build_cnn_model(
        w2v_embedding_matrix, EMBEDDING_DIM_W2V, MAX_LEN,
        name="CNN_W2V_70_30", trainable_embeddings=True
    )
    hist, loss, acc = train_and_evaluate(model_w2v_70, X_train_70, X_test_70, y_train_70, y_test_70)
    results["Word2Vec | 70/30"] = {"loss": loss, "accuracy": acc}

    #Save results
    results_df = pd.DataFrame(results).T
    results_df.index.name = "Experiment"
    results_df = results_df.round(4)
    results_df.to_csv(f"results_maxlen_{MAX_LEN}.csv")
    print(f"\nSaved: results_maxlen_{MAX_LEN}.csv")
    print(results_df.to_string())



██████████████████████████████████████████████████████████████████████
  MAX_LEN = 500
██████████████████████████████████████████████████████████████████████
  GloVe-50d  |  80/20 Split  |  MAX_LEN=500


/Users/applecare/miniconda3/lib/python3.13/site-packages/keras/src/layers/core/embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "CNN_GloVe_80_20"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 500)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 500, 50)        │     3,765,150 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 500, 128)       │        32,128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 500, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 250, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 250, 128)       │        82,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 250, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 125, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ (None, 128)            │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,897,636 (14.87 MB)

 Trainable params: 131,974 (515.52 KB)

 Non-trainable params: 3,765,662 (14.36 MB)

Epoch 1/5
  70/4690 ━━━━━━━━━━━━━━━━━━━━ 7:21 96ms/step - accuracy: 0.2786 - loss: 5.2905

KeyboardInterrupt: 

4.4 — Results Comparison Table

In [ ]:
results_df = pd.DataFrame(results).T
results_df.index.name = "Experiment"
results_df = results_df.round(4)
print("\n")
print(results_df.to_string())
